# Beam tracking from many H5 files

This notebook processes one H5 file at a time. Each file is treated as one frame.

Expected H5 structure, based on your inspection:

```text
/data : 512 x 512 detector image
```

The notebook extracts only numerical tracking results and does not keep all images in memory.


## Slow version

This version fits the cropped beam area with a full 2D Gaussian or 2D pseudo-Voigt approximation.

In [ ]:
# %% Imports
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter, center_of_mass
from scipy.optimize import least_squares

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

In [ ]:
# %% Folder and file list

data_folder = Path(r"C:\Users\missagli\Documents\RAW_cut1")

h5_files = list(data_folder.glob("*.h5"))


def natural_key(path):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", path.name)]


h5_files = sorted(h5_files, key=natural_key)

print("Number of H5 files:", len(h5_files))
if len(h5_files) > 0:
    print("First file:", h5_files[0])
    print("Last file:", h5_files[-1])

In [ ]:
# %% Inspect one H5 file

test_file = h5_files[0]
print("Testing:", test_file)

with h5py.File(test_file, "r") as f:
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"{name}: shape={obj.shape}, dtype={obj.dtype}")
        else:
            print(f"{name}/")

    f.visititems(visitor)

    print("\nRoot attributes:")
    for k, v in f.attrs.items():
        print(k, "=", v)

    if "data" in f:
        print("\nDataset /data attributes:")
        for k, v in f["data"].attrs.items():
            print(k, "=", v)

In [ ]:
# %% H5 reader

image_path = "data"

metadata_names = [
    "Pressure",
    "Temperature_A",
    "Temperature_B",
    "Time",
    "Time_for_humans",
    "Delay_ps",
    "RF_Cavity_phase",
    "RF_Cavity_power",
    "UV_power",
    "pump_power",
    "shutter",
]


def clean_h5_value(value):
    """Convert H5 scalars/bytes into convenient Python objects."""
    if isinstance(value, bytes):
        return value.decode(errors="ignore")

    if isinstance(value, np.ndarray):
        if value.shape == ():
            return clean_h5_value(value.item())
        if value.dtype.kind == "S":
            return value.astype(str)

    return value


def read_one_h5(path):
    """
    Read one H5 file.

    Returns
    -------
    image : 2D float32 array
    meta : dict
    """
    with h5py.File(path, "r") as f:
        image = f[image_path][()]

        meta = {}
        for name in metadata_names:
            if name in f.attrs:
                meta[name] = clean_h5_value(f.attrs[name])
            elif name in f[image_path].attrs:
                meta[name] = clean_h5_value(f[image_path].attrs[name])
            else:
                meta[name] = np.nan

    image = np.squeeze(image)

    if image.ndim != 2:
        raise ValueError(f"Expected 2D image, got shape {image.shape}")

    # Filenames look like timestamps, e.g. 1782742652.079.h5
    try:
        meta["file_timestamp"] = float(Path(path).stem)
    except ValueError:
        meta["file_timestamp"] = np.nan

    return image.astype(np.float32, copy=False), meta

In [ ]:
# %% Test reader and display first image

image, meta = read_one_h5(h5_files[0])

print("Image shape:", image.shape)
print("Image min/max:", np.nanmin(image), np.nanmax(image))
print(meta)

plt.figure(figsize=(6, 5))
plt.imshow(np.log1p(image), origin="lower", cmap="gray")
plt.colorbar(label="log(1 + counts)")
plt.xlabel("x pixel")
plt.ylabel("y pixel")
plt.title(h5_files[0].name)
plt.tight_layout()
plt.show()

In [ ]:
# %% Plot helpers

def robust_ylim(y, q_low=0.5, q_high=99.5, margin_frac=0.10):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]

    if y.size == 0:
        return 0, 1

    ymin, ymax = np.percentile(y, [q_low, q_high])

    if not np.isfinite(ymin) or not np.isfinite(ymax):
        return 0, 1

    if ymax == ymin:
        margin = 1.0 if ymax == 0 else 0.1 * abs(ymax)
    else:
        margin = margin_frac * (ymax - ymin)

    return ymin - margin, ymax + margin


def robust_ylim_multi(arrays, q_low=0.5, q_high=99.5, margin_frac=0.10):
    values = []
    for arr in arrays:
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size > 0:
            values.append(arr)

    if not values:
        return 0, 1

    y = np.concatenate(values)
    return robust_ylim(y, q_low=q_low, q_high=q_high, margin_frac=margin_frac)


def find_existing_column(candidates, df):
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"None of these columns found: {candidates}")

In [ ]:
# %% Tracking and fitting parameters

# Change this after looking at the model comparison table.
chosen_model = "gaussian"     # "gaussian" or "pvoigt"

# ROI used for the moving fit around the beam.
fit_roi_half_size = 80

# Used only for finding the initial center and robust guesses.
smooth_sigma = 1.0
background_percentile = 10
threshold_rel = 0.10

# Counts inside fitted ellipse: 1.0 means inside the FWHM ellipse.
fit_count_scale = 1.0

# Fixed ROI based on the first frame.
# Use a generous ROI so the beam stays inside even if it drifts.
fixed_roi_scale = 3.0
fixed_extra_pix_x = 20
fixed_extra_pix_y = 20

scatter_size = 3
scatter_alpha = 0.7

In [ ]:
# %% 2D Gaussian and pseudo-Voigt models

LN2 = np.log(2)


def gaussian_2d(params, xx, yy):
    offset, amp, x0, y0, fwhm_x, fwhm_y = params

    ux = (xx - x0) / (fwhm_x / 2)
    uy = (yy - y0) / (fwhm_y / 2)

    return offset + amp * np.exp(-LN2 * (ux**2 + uy**2))


def pseudo_voigt_2d(params, xx, yy):
    """
    2D pseudo-Voigt approximation.

    A true Voigt is a convolution of a Gaussian and Lorentzian.
    This pseudo-Voigt is a weighted sum and is much faster.
    """
    offset, amp, x0, y0, fwhm_x, fwhm_y, eta = params

    ux = (xx - x0) / (fwhm_x / 2)
    uy = (yy - y0) / (fwhm_y / 2)

    r2 = ux**2 + uy**2

    gaussian_part = np.exp(-LN2 * r2)
    lorentzian_part = 1 / (1 + r2)

    profile = (1 - eta) * gaussian_part + eta * lorentzian_part

    return offset + amp * profile

In [ ]:
# %% Slow full 2D fit on a cropped ROI

def fit_beam_image_2d(
    image,
    model_name="gaussian",
    previous_center=None,
    fit_roi_half_size=80,
    smooth_sigma=1.0,
    background_percentile=10,
    threshold_rel=0.10,
    use_poisson_weights=True,
):
    image = np.asarray(image, dtype=np.float32)
    image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)

    ny, nx = image.shape

    if previous_center is None:
        smoothed = gaussian_filter(image, smooth_sigma)
        y_guess, x_guess = np.unravel_index(np.argmax(smoothed), smoothed.shape)
    else:
        x_guess, y_guess = previous_center

    x_min = max(0, int(round(x_guess)) - fit_roi_half_size)
    x_max = min(nx, int(round(x_guess)) + fit_roi_half_size + 1)

    y_min = max(0, int(round(y_guess)) - fit_roi_half_size)
    y_max = min(ny, int(round(y_guess)) + fit_roi_half_size + 1)

    crop = image[y_min:y_max, x_min:x_max]
    yy, xx = np.indices(crop.shape)

    bg0 = np.percentile(crop, background_percentile)
    weights = crop - bg0
    weights[weights < 0] = 0
    weights = gaussian_filter(weights, smooth_sigma)

    max_w = np.max(weights)

    if max_w > 0:
        weights_thr = weights.copy()
        weights_thr[weights_thr < threshold_rel * max_w] = 0

        if np.sum(weights_thr) <= 0:
            weights_thr = weights

        y0_init, x0_init = center_of_mass(weights_thr)

        if not np.isfinite(x0_init) or not np.isfinite(y0_init):
            y0_init, x0_init = np.unravel_index(np.argmax(weights), weights.shape)

        amp0 = np.max(crop) - bg0

        total_w = np.sum(weights_thr)
        if total_w > 0:
            sx = np.sqrt(np.sum(weights_thr * (xx - x0_init)**2) / total_w)
            sy = np.sqrt(np.sum(weights_thr * (yy - y0_init)**2) / total_w)
            fwhm_x0 = max(4, 2.355 * sx)
            fwhm_y0 = max(4, 2.355 * sy)
        else:
            fwhm_x0 = fit_roi_half_size / 2
            fwhm_y0 = fit_roi_half_size / 2

    else:
        amp0 = np.max(crop) - bg0
        y0_init, x0_init = np.unravel_index(np.argmax(crop), crop.shape)
        fwhm_x0 = fit_roi_half_size / 2
        fwhm_y0 = fit_roi_half_size / 2

    lower_common = [0, 0, 0, 0, 2, 2]
    upper_common = [
        np.inf,
        np.inf,
        crop.shape[1] - 1,
        crop.shape[0] - 1,
        2 * fit_roi_half_size,
        2 * fit_roi_half_size,
    ]

    if model_name == "gaussian":
        p0 = [bg0, amp0, x0_init, y0_init, fwhm_x0, fwhm_y0]
        lower = lower_common
        upper = upper_common
        model_fun = gaussian_2d

    elif model_name == "pvoigt":
        p0 = [bg0, amp0, x0_init, y0_init, fwhm_x0, fwhm_y0, 0.5]
        lower = lower_common + [0]
        upper = upper_common + [1]
        model_fun = pseudo_voigt_2d

    else:
        raise ValueError("model_name must be 'gaussian' or 'pvoigt'")

    def residuals(p):
        model = model_fun(p, xx, yy)

        if use_poisson_weights:
            sigma = np.sqrt(np.maximum(crop, 1))
            return ((model - crop) / sigma).ravel()

        return (model - crop).ravel()

    result = least_squares(
        residuals,
        p0,
        bounds=(lower, upper),
        loss="soft_l1",
        max_nfev=3000,
        x_scale="jac",
    )

    p = result.x
    model_crop = model_fun(p, xx, yy)
    residual_crop = crop - model_crop

    rss = np.sum((crop - model_crop)**2)
    n = crop.size
    k = len(p)
    rmse = np.sqrt(rss / n)

    if rss > 0:
        aic = n * np.log(rss / n) + 2 * k
        bic = n * np.log(rss / n) + k * np.log(n)
    else:
        aic = -np.inf
        bic = -np.inf

    sst = np.sum((crop - np.mean(crop))**2)
    r2 = 1 - rss / sst if sst > 0 else np.nan

    x_center = x_min + p[2]
    y_center = y_min + p[3]

    return {
        "model": model_name,
        "success": result.success,
        "message": result.message,
        "offset": p[0],
        "amplitude": p[1],
        "x_pix": x_center,
        "y_pix": y_center,
        "fwhm_x_pix": p[4],
        "fwhm_y_pix": p[5],
        "eta": p[6] if model_name == "pvoigt" else np.nan,
        "rmse": rmse,
        "aic": aic,
        "bic": bic,
        "r2": r2,
        "crop": crop,
        "model_crop": model_crop,
        "residual_crop": residual_crop,
        "crop_origin": (x_min, y_min),
    }

In [ ]:
# %% 1D profile models, used by the fast method and by fixed-ROI FWHM

LN2 = np.log(2)


def gaussian_1d(p, x):
    offset, amp, x0, fwhm = p
    u = (x - x0) / (fwhm / 2)
    return offset + amp * np.exp(-LN2 * u**2)


def pvoigt_1d(p, x):
    offset, amp, x0, fwhm, eta = p
    u = (x - x0) / (fwhm / 2)

    gaussian_part = np.exp(-LN2 * u**2)
    lorentzian_part = 1 / (1 + u**2)

    return offset + amp * ((1 - eta) * gaussian_part + eta * lorentzian_part)


def fit_profile_1d(profile, model_name="gaussian"):
    profile = np.asarray(profile, dtype=np.float32)
    x = np.arange(profile.size, dtype=np.float32)

    offset0 = np.percentile(profile, 10)
    amp0 = np.max(profile) - offset0
    x0_0 = np.argmax(profile)

    halfmax = offset0 + amp0 / 2
    above = np.where(profile > halfmax)[0]

    if len(above) > 2:
        fwhm0 = above[-1] - above[0]
    else:
        fwhm0 = profile.size / 4

    fwhm0 = max(3, fwhm0)

    if model_name == "gaussian":
        p0 = [offset0, amp0, x0_0, fwhm0]
        lower = [-np.inf, 0, 0, 2]
        upper = [np.inf, np.inf, profile.size - 1, profile.size]
        model_fun = gaussian_1d

    elif model_name == "pvoigt":
        p0 = [offset0, amp0, x0_0, fwhm0, 0.5]
        lower = [-np.inf, 0, 0, 2, 0]
        upper = [np.inf, np.inf, profile.size - 1, profile.size, 1]
        model_fun = pvoigt_1d

    else:
        raise ValueError("model_name must be 'gaussian' or 'pvoigt'")

    def residuals(p):
        return model_fun(p, x) - profile

    result = least_squares(
        residuals,
        p0,
        bounds=(lower, upper),
        loss="soft_l1",
        max_nfev=300,
        x_scale="jac",
    )

    p = result.x
    fit = model_fun(p, x)
    rss = np.sum((profile - fit)**2)
    rmse = np.sqrt(rss / profile.size)

    return {
        "success": result.success,
        "offset": p[0],
        "amplitude": p[1],
        "center": p[2],
        "fwhm": p[3],
        "eta": p[4] if model_name == "pvoigt" else np.nan,
        "rmse": rmse,
        "fit": fit,
    }

In [ ]:
# %% Counts and fixed-ROI FWHM helpers

def counts_in_fixed_roi(image, fixed_center_x, fixed_center_y, fixed_half_x, fixed_half_y):
    ny, nx = image.shape

    x0 = max(0, int(round(fixed_center_x)) - fixed_half_x)
    x1 = min(nx, int(round(fixed_center_x)) + fixed_half_x + 1)

    y0 = max(0, int(round(fixed_center_y)) - fixed_half_y)
    y1 = min(ny, int(round(fixed_center_y)) + fixed_half_y + 1)

    roi = image[y0:y1, x0:x1]

    raw_counts = np.sum(roi)

    bg = np.percentile(roi, background_percentile)
    bgsub_counts = np.sum(np.clip(roi - bg, 0, None))

    return raw_counts, bgsub_counts, roi.size, (x0, x1, y0, y1)


def fwhm_from_fixed_roi_profiles(
    image,
    fixed_center_x,
    fixed_center_y,
    fixed_half_x,
    fixed_half_y,
    model_name="gaussian",
):
    """
    Compute FWHM x/y from the same fixed rectangular ROI for every frame.

    The ROI does not move or resize after it is defined from the first frame.
    The fitted width inside the ROI is still allowed to change.
    """
    ny, nx = image.shape

    x0 = max(0, int(round(fixed_center_x)) - fixed_half_x)
    x1 = min(nx, int(round(fixed_center_x)) + fixed_half_x + 1)

    y0 = max(0, int(round(fixed_center_y)) - fixed_half_y)
    y1 = min(ny, int(round(fixed_center_y)) + fixed_half_y + 1)

    roi = image[y0:y1, x0:x1]

    profile_x = np.sum(roi, axis=0)
    profile_y = np.sum(roi, axis=1)

    fit_x = fit_profile_1d(profile_x, model_name=model_name)
    fit_y = fit_profile_1d(profile_y, model_name=model_name)

    fixed_x_pix = x0 + fit_x["center"]
    fixed_y_pix = y0 + fit_y["center"]

    return {
        "fixed_x_pix": fixed_x_pix,
        "fixed_y_pix": fixed_y_pix,
        "fixed_fwhm_x_pix": fit_x["fwhm"],
        "fixed_fwhm_y_pix": fit_y["fwhm"],
        "fixed_eta_x": fit_x["eta"],
        "fixed_eta_y": fit_y["eta"],
        "fixed_rmse_x": fit_x["rmse"],
        "fixed_rmse_y": fit_y["rmse"],
        "fixed_roi_bounds": (x0, x1, y0, y1),
    }

In [ ]:
# %% Slow fit-defined ellipse counts

def counts_in_fitted_ellipse(image, fit, scale=1.0):
    ny, nx = image.shape

    x0 = fit["x_pix"]
    y0 = fit["y_pix"]

    rx = max(scale * fit["fwhm_x_pix"] / 2, 1)
    ry = max(scale * fit["fwhm_y_pix"] / 2, 1)

    # Only create a mask in a local bounding box.
    xmin = max(0, int(np.floor(x0 - rx)))
    xmax = min(nx, int(np.ceil(x0 + rx)) + 1)

    ymin = max(0, int(np.floor(y0 - ry)))
    ymax = min(ny, int(np.ceil(y0 + ry)) + 1)

    roi = image[ymin:ymax, xmin:xmax]
    yy, xx = np.indices(roi.shape)
    xx = xx + xmin
    yy = yy + ymin

    mask = ((xx - x0) / rx)**2 + ((yy - y0) / ry)**2 <= 1

    raw_counts = np.sum(roi[mask])
    bgsub_counts = np.sum(np.clip(roi[mask] - fit["offset"], 0, None))

    return raw_counts, bgsub_counts, np.sum(mask)

In [ ]:
# %% Compare Gaussian and pseudo-Voigt on selected files

selected_indices = sorted(set([0, len(h5_files) // 2, len(h5_files) - 1]))

comparison_rows = []
fit_checks = {}

for idx in selected_indices:
    path = h5_files[idx]
    image, meta = read_one_h5(path)

    for model_name in ["gaussian", "pvoigt"]:
        fit = fit_beam_image_2d(
            image,
            model_name=model_name,
            previous_center=None,
            fit_roi_half_size=fit_roi_half_size,
            smooth_sigma=smooth_sigma,
            background_percentile=background_percentile,
            threshold_rel=threshold_rel,
        )

        comparison_rows.append({
            "file_index": idx,
            "filename": path.name,
            "model": model_name,
            "success": fit["success"],
            "x_pix": fit["x_pix"],
            "y_pix": fit["y_pix"],
            "fwhm_x_pix": fit["fwhm_x_pix"],
            "fwhm_y_pix": fit["fwhm_y_pix"],
            "eta": fit["eta"],
            "rmse": fit["rmse"],
            "aic": fit["aic"],
            "bic": fit["bic"],
            "r2": fit["r2"],
        })

        fit_checks[(idx, model_name)] = fit

comparison = pd.DataFrame(comparison_rows)
display(comparison)

display(comparison.groupby("model")[["rmse", "aic", "bic", "r2"]].median())

chosen_model = comparison.groupby("model")["bic"].median().idxmin()
print("Chosen model based on median BIC:", chosen_model)

# Override manually if needed:
# chosen_model = "gaussian"
# chosen_model = "pvoigt"

In [ ]:
# %% Plot selected 2D fits and residuals

for idx in selected_indices:
    for model_name in ["gaussian", "pvoigt"]:
        fit = fit_checks[(idx, model_name)]

        crop = fit["crop"]
        model_crop = fit["model_crop"]
        residual_crop = fit["residual_crop"]

        vmax = np.percentile(np.log1p(crop), 99.5)

        fig, axes = plt.subplots(1, 3, figsize=(13, 4))

        axes[0].imshow(np.log1p(crop), origin="lower", cmap="gray", vmax=vmax)
        axes[0].set_title("Data")

        axes[1].imshow(np.log1p(model_crop), origin="lower", cmap="gray", vmax=vmax)
        axes[1].set_title(f"{model_name} fit")

        im = axes[2].imshow(residual_crop, origin="lower", cmap="coolwarm")
        axes[2].set_title("Residual")
        plt.colorbar(im, ax=axes[2], fraction=0.046)

        fig.suptitle(
            f"{h5_files[idx].name} — {model_name} — "
            f"FWHM x={fit['fwhm_x_pix']:.1f} px, y={fit['fwhm_y_pix']:.1f} px, "
            f"BIC={fit['bic']:.1f}"
        )

        plt.tight_layout()
        plt.show()

In [ ]:
# %% Define fixed ROI from first frame

first_image, first_meta = read_one_h5(h5_files[0])

first_fit = fit_beam_image_2d(
    first_image,
    model_name=chosen_model,
    previous_center=None,
    fit_roi_half_size=fit_roi_half_size,
    smooth_sigma=smooth_sigma,
    background_percentile=background_percentile,
    threshold_rel=threshold_rel,
)

fixed_center_x = first_fit["x_pix"]
fixed_center_y = first_fit["y_pix"]

fixed_half_x = int(np.ceil(fixed_roi_scale * first_fit["fwhm_x_pix"] / 2 + fixed_extra_pix_x))
fixed_half_y = int(np.ceil(fixed_roi_scale * first_fit["fwhm_y_pix"] / 2 + fixed_extra_pix_y))

print("Fixed ROI center:", fixed_center_x, fixed_center_y)
print("Fixed ROI half-size x/y:", fixed_half_x, fixed_half_y)

In [ ]:
# %% Main loop — slow 2D fit on every file

results_rows = []
previous_center = None

for frame_index, path in enumerate(tqdm(h5_files)):
    image, meta = read_one_h5(path)

    fit = fit_beam_image_2d(
        image,
        model_name=chosen_model,
        previous_center=previous_center,
        fit_roi_half_size=fit_roi_half_size,
        smooth_sigma=smooth_sigma,
        background_percentile=background_percentile,
        threshold_rel=threshold_rel,
    )

    if fit["success"] and np.isfinite(fit["x_pix"]) and np.isfinite(fit["y_pix"]):
        previous_center = (fit["x_pix"], fit["y_pix"])

    fit_counts_raw, fit_counts_bgsub, fit_area_pixels = counts_in_fitted_ellipse(
        image,
        fit,
        scale=fit_count_scale,
    )

    fixed_counts_raw, fixed_counts_bgsub, fixed_area_pixels, fixed_roi_bounds = counts_in_fixed_roi(
        image,
        fixed_center_x,
        fixed_center_y,
        fixed_half_x,
        fixed_half_y,
    )

    fixed_width = fwhm_from_fixed_roi_profiles(
        image,
        fixed_center_x,
        fixed_center_y,
        fixed_half_x,
        fixed_half_y,
        model_name=chosen_model,
    )

    row = {
        "frame": frame_index,
        "filename": path.name,
        "file_timestamp": meta.get("file_timestamp", np.nan),
        "model": chosen_model,
        "success": fit["success"],

        "x_pix": fit["x_pix"],
        "y_pix": fit["y_pix"],
        "fwhm_x_pix": fit["fwhm_x_pix"],
        "fwhm_y_pix": fit["fwhm_y_pix"],
        "eta": fit["eta"],

        "fixed_x_pix": fixed_width["fixed_x_pix"],
        "fixed_y_pix": fixed_width["fixed_y_pix"],
        "fixed_fwhm_x_pix": fixed_width["fixed_fwhm_x_pix"],
        "fixed_fwhm_y_pix": fixed_width["fixed_fwhm_y_pix"],
        "fixed_eta_x": fixed_width["fixed_eta_x"],
        "fixed_eta_y": fixed_width["fixed_eta_y"],
        "fixed_rmse_x": fixed_width["fixed_rmse_x"],
        "fixed_rmse_y": fixed_width["fixed_rmse_y"],

        "offset": fit["offset"],
        "amplitude": fit["amplitude"],
        "rmse": fit["rmse"],
        "aic": fit["aic"],
        "bic": fit["bic"],
        "r2": fit["r2"],

        "fit_counts_raw": fit_counts_raw,
        "fit_counts_bgsub": fit_counts_bgsub,
        "fit_area_pixels": fit_area_pixels,

        "fixed_counts_raw": fixed_counts_raw,
        "fixed_counts_bgsub": fixed_counts_bgsub,
        "fixed_area_pixels": fixed_area_pixels,
    }

    row.update(meta)
    results_rows.append(row)

tracking = pd.DataFrame(results_rows)

tracking["dx_pix"] = tracking["x_pix"] - tracking["x_pix"].iloc[0]
tracking["dy_pix"] = tracking["y_pix"] - tracking["y_pix"].iloc[0]
tracking["fixed_dx_pix"] = tracking["fixed_x_pix"] - tracking["fixed_x_pix"].iloc[0]
tracking["fixed_dy_pix"] = tracking["fixed_y_pix"] - tracking["fixed_y_pix"].iloc[0]
tracking["time_s"] = tracking["file_timestamp"] - tracking["file_timestamp"].iloc[0]

tracking.head()

In [ ]:
# %% Save results

output_csv = data_folder / "beam_tracking_results_slow_2d.csv"
output_pkl = data_folder / "beam_tracking_results_slow_2d.pkl"

tracking.to_csv(output_csv, index=False)
tracking.to_pickle(output_pkl)

print("Saved:", output_csv)
print("Saved:", output_pkl)

In [ ]:
# %% Plot beam movement

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].scatter(tracking["frame"], tracking["dx_pix"], s=scatter_size, alpha=scatter_alpha)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel("Frame")
axes[0].set_ylabel(r"$\Delta x$ pixel")
axes[0].set_title("Beam movement along x")
axes[0].set_ylim(robust_ylim(tracking["dx_pix"]))
axes[0].grid(True)

axes[1].scatter(tracking["frame"], tracking["dy_pix"], s=scatter_size, alpha=scatter_alpha)
axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel(r"$\Delta y$ pixel")
axes[1].set_title("Beam movement along y")
axes[1].set_ylim(robust_ylim(tracking["dy_pix"]))
axes[1].grid(True)

fig.suptitle("Central beam movement")
plt.tight_layout()
plt.show()

In [ ]:
# %% Plot beam size from fixed ROI only

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].scatter(tracking["frame"], tracking["fixed_fwhm_x_pix"], s=scatter_size, alpha=scatter_alpha)
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("FWHM x pixel")
axes[0].set_title("Beam size along x — fixed ROI")
axes[0].set_ylim(robust_ylim(tracking["fixed_fwhm_x_pix"]))
axes[0].grid(True)

axes[1].scatter(tracking["frame"], tracking["fixed_fwhm_y_pix"], s=scatter_size, alpha=scatter_alpha)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("FWHM y pixel")
axes[1].set_title("Beam size along y — fixed ROI")
axes[1].set_ylim(robust_ylim(tracking["fixed_fwhm_y_pix"]))
axes[1].grid(True)

fig.suptitle("Central beam size from fixed ROI")
plt.tight_layout()
plt.show()

In [ ]:
# %% Compare moving fit ROI FWHM and fixed ROI FWHM

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].scatter(
    tracking["frame"],
    tracking["fwhm_x_pix"],
    s=scatter_size,
    alpha=scatter_alpha,
    label="Moving fit ROI",
)
axes[0].scatter(
    tracking["frame"],
    tracking["fixed_fwhm_x_pix"],
    s=scatter_size,
    alpha=scatter_alpha,
    label="Fixed ROI",
)
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("FWHM x pixel")
axes[0].set_title("Beam size along x")
axes[0].set_ylim(robust_ylim_multi([tracking["fwhm_x_pix"], tracking["fixed_fwhm_x_pix"]]))
axes[0].grid(True)
axes[0].legend()

axes[1].scatter(
    tracking["frame"],
    tracking["fwhm_y_pix"],
    s=scatter_size,
    alpha=scatter_alpha,
    label="Moving fit ROI",
)
axes[1].scatter(
    tracking["frame"],
    tracking["fixed_fwhm_y_pix"],
    s=scatter_size,
    alpha=scatter_alpha,
    label="Fixed ROI",
)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("FWHM y pixel")
axes[1].set_title("Beam size along y")
axes[1].set_ylim(robust_ylim_multi([tracking["fwhm_y_pix"], tracking["fixed_fwhm_y_pix"]]))
axes[1].grid(True)
axes[1].legend()

fig.suptitle("Central beam size: moving ROI vs fixed ROI")
plt.tight_layout()
plt.show()

In [ ]:
# %% Plot counts

fit_count_col = find_existing_column(["fit_counts_bgsub", "fit_counts_raw"], tracking)
fixed_count_col = find_existing_column(["fixed_counts_bgsub", "fixed_counts_raw"], tracking)

print("Using fit-count column:", fit_count_col)
print("Using fixed-ROI column:", fixed_count_col)

y_fit = pd.to_numeric(tracking[fit_count_col], errors="coerce")
y_fixed = pd.to_numeric(tracking[fixed_count_col], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].scatter(tracking["frame"], y_fit, s=scatter_size, alpha=scatter_alpha)
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Counts")
axes[0].set_title("Counts in fit-defined beam area")
axes[0].set_ylim(robust_ylim(y_fit))
axes[0].grid(True)

axes[1].scatter(tracking["frame"], y_fixed, s=scatter_size, alpha=scatter_alpha)
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Counts")
axes[1].set_title("Counts in fixed ROI")
axes[1].set_ylim(robust_ylim(y_fixed))
axes[1].grid(True)

fig.suptitle("Central beam counts")
plt.tight_layout()
plt.show()

In [ ]:
# %% Plot pressure and temperatures if they are available

available_meta = []
for col in ["Pressure", "Temperature_A", "Temperature_B"]:
    if col in tracking.columns:
        y = pd.to_numeric(tracking[col], errors="coerce")
        if np.isfinite(y).any():
            available_meta.append(col)

if not available_meta:
    print("No pressure/temperature metadata found in these H5 files.")
else:
    fig, axes = plt.subplots(len(available_meta), 1, figsize=(8, 3 * len(available_meta)), sharex=True)

    if len(available_meta) == 1:
        axes = [axes]

    for ax, col in zip(axes, available_meta):
        y = pd.to_numeric(tracking[col], errors="coerce")
        ax.scatter(tracking["frame"], y, s=scatter_size, alpha=scatter_alpha)
        ax.set_ylabel(col)
        ax.set_ylim(robust_ylim(y))
        ax.grid(True)

    axes[-1].set_xlabel("Frame")
    fig.suptitle("Metadata vs frame")
    plt.tight_layout()
    plt.show()